# Local dataset audit / 本地数据检查

This template reads authorised local OptionMetrics files. Keep this tracked
template output-free. Execute it only into the ignored `.local-notebooks/`
directory.

本模板读取获得授权的本地 OptionMetrics 文件。提交的模板必须保持无输出，并且
只能将执行结果保存到被忽略的 `.local-notebooks/` 目录。

In [ ]:
from pathlib import Path
import csv
import itertools
import json
from IPython.display import display


def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


ROOT = find_repository_root()
DATA_DIR = ROOT / Path("data")
PROFILE_PATH = ROOT / "docs" / "data" / "dataset_profile.json"
EXPECTED_FILES = [
    "Index Dividend Yield.csv",
    "Zero Coupon Yield Curve.csv",
    "option price.csv",
    "security price.csv",
]

## Local file status / 本地文件状态

In [ ]:
file_status = []
for name in EXPECTED_FILES:
    path = DATA_DIR / name
    file_status.append({
        "file": name,
        "present": path.is_file(),
        "size_gib": round(path.stat().st_size / (1024 ** 3), 3) if path.is_file() else None,
    })
display(file_status)

## Five-row licensed preview / 五行真实数据预览

The output below is licensed and must never be committed.

以下输出受数据许可保护，严禁提交。

In [ ]:
preview = {}
for name in EXPECTED_FILES:
    path = DATA_DIR / name
    if not path.is_file():
        preview[name] = {"error": "missing local file"}
        continue
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        preview[name] = list(itertools.islice(reader, 5))
display(preview)

## Aggregate profile / 汇总概况

In [ ]:
profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
summary = {
    name: {
        "rows": table["row_count"],
        "date_range": table["date_range"],
        "columns": len(table["columns"]),
    }
    for name, table in profile["tables"].items()
}
display(summary)